In [ ]:
import pandas as pd

# Load your UCI training compounds
df = pd.read_csv("train_raw.csv")
df_ext  = pd.read_csv("Stanev_External_Validation_Final.csv")

# Simple approach: remove exact Tc duplicates
uci_tc_values = set(df["critical_temp"].round(2))

# Filter out the overlap and save to df_external
df_external = df_ext[~df_ext["Tc"].round(2).isin(uci_tc_values)].copy()

# Print the length of df_external
print(f"Original Stanev size: {len(df_ext)}")
print(f"External validation set size (after dropping overlap): {len(df_external)}")
df_external.to_csv("stanev_250_unseen_compounds.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

# Load 250 unseen compounds file
df_250 = pd.read_csv("stanev_250_unseen_compounds.csv") 

print("250 COMPOUND Tc DISTRIBUTION:")
bins = [(0,10),(10,30),(30,50),(50,80),(80,100),(100,135)]
for lo,hi in bins:
    n = ((df_250['Tc']>=lo)&(df_250['Tc']<hi)).sum()
    pct = 100*n/len(df_250)
    print(f"  {lo}-{hi}K: {n} ({pct:.1f}%)")
print(f"\nMean Tc: {df_250['Tc'].mean():.2f} K")
print(f"Max Tc:  {df_250['Tc'].max():.2f} K")
print(f"Std Tc:  {df_250['Tc'].std():.2f} K")

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import joblib

# 1. Load my saved model and scaler
my_model = joblib.load("stacking_model_50features.pkl")
scaler     = joblib.load("standard_scaler.pkl")

# 2. Get my top 50 feature names again (just to be safe)
df_top50 = pd.read_csv("top50_train.csv")
top50_cols = [c for c in df_top50.columns if c != "critical_temp"]

# 3. Handle NaNs the EXACT same way we did in training (fillna with 0)
# We fill NaNs before converting to .values so Pandas handles it cleanly
df_external_clean = df_external.copy()
df_external_clean[top50_cols] = df_external_clean[top50_cols].fillna(0)

# 4. Prepare features and target
X_ext = df_external_clean[top50_cols].values
y_ext = df_external_clean["Tc"].values

# 5. Scale using my scaler (using .transform() only!)
X_ext_scaled = scaler.transform(X_ext)

# 6. Predict!
y_pred = my_model.predict(X_ext_scaled)

# print the final metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

r2 = r2_score(y_ext, y_pred)
rmse = np.sqrt(mean_squared_error(y_ext, y_pred))
mae = mean_absolute_error(y_ext, y_pred)

print(f"--- FINAL TRUE EXTERNAL VALIDATION ---")
print(f"Compounds tested: {len(y_ext)}")
print(f"R²:   {r2:.4f}")
print(f"RMSE: {rmse:.4f} K")
print(f"MAE:  {mae:.4f} K")
